# Bronze Layer

# Step 1 - Define Schema as String

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("discount", StringType(), True),
    StructField("total_amount", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("store_location", StringType(), True),
    StructField("order_status", StringType(), True)
])

customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("date_of_birth", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("registration_date", StringType(), True),
    StructField("loyalty_points", StringType(), True),
    StructField("status", StringType(), True)
])

# Step 2- Read raw CSV files

In [0]:
order_broze_df = spark.read.format("csv").option("header", "true").schema(orders_schema).load("/Volumes/retailproject/raw_data/raw_file/retail_orders_messy.csv")

customer_broze_df = spark.read.format("csv").option("header", "true").schema(customer_schema).load("/Volumes/retailproject/raw_data/raw_file/retail_customers_messy.csv")


In [0]:
order_broze_df.display()
customer_broze_df.display()


# Step 3- Add ingestion MetaData

In [0]:
from pyspark.sql.functions import current_timestamp
order_broze_df = order_broze_df.withColumn("ingest_time", current_timestamp())
customer_broze_df = customer_broze_df.withColumn("ingest_time", current_timestamp())

order_broze_df.display()
customer_broze_df.display()

In [0]:
order_broze_df.explain("formatted")

# Step4- Save a Delta Table in Bronze Layer

In [0]:


order_broze_df.write.format("delta").mode("overwrite").saveAsTable('retailproject.bronze.bronze_orders')
customer_broze_df.write.format("delta").mode("overwrite").saveAsTable('retailproject.bronze.bronze_customers')

# Step 1- Load Bronze Orders

In [0]:
bronze_orders_df = spark.read.table("retailproject.bronze.bronze_orders")
display(bronze_orders_df)

# Step 2- Fix date format

In [0]:
from pyspark.sql.functions import col, to_date, try_to_date, coalesce

silver_order_df =bronze_orders_df.withColumn ('order_date',
                                              coalesce(
                                                  try_to_date(col('order_date'), 'MM/dd/yyyy'),
                                                  try_to_date(col('order_date'), 'yyyy-MM-dd'),
                                                  try_to_date(col('order_date'), 'MM-dd-yyyy'),
                                                  try_to_date(col('order_date'), 'dd-MM-yyyy'),
                                              ))

silver_order_df.display()

In [0]:
from pyspark.sql import functions as F

orders = spark.table("retailproject.bronze.bronze_orders")

key_counts = (
    orders
    .groupBy("customer_id")
    .count()
    .orderBy(F.desc("count"))
)

display(key_counts.limit(20))

# Step 3- Standardize category value

In [0]:
silver_order_df.select("category").distinct().display()


In [0]:
from pyspark.sql.functions import upper,trim,when

silver_order_df = silver_order_df.withColumn('category', upper(trim(col('category'))))

silver_order_df = silver_order_df.withColumn(
                                             'category', when(col('category') == 'ELECTRONIC','ELECTRONICS').
                                             when(col('category')=='electronics','ELECTRONICS').
                                             when(col('category') =='FASHON','FASHION').
                                             when(col('category') =='HOME AND LIVING','HOME & LIVING').otherwise(col('category')))
silver_order_df.display()


# Step 4- Remove duplicate orders

In [0]:
from pyspark.sql.functions import count, countDistinct

silver_order_df.select(count("order_id"), countDistinct("order_id")).display()

In [0]:
silver_order_df = silver_order_df.dropDuplicates(['order_id'])


# Step5 - Handle Null values discount

In [0]:
from pyspark.sql.functions import when, col

silver_order_df = silver_order_df = silver_order_df.withColumn('discount',
                                 when(col('discount').isNull(),'0').when(col('discount')=='ten','0.1').otherwise(col('discount'))
                                 
                                 )

In [0]:
silver_order_df.display()

#  STEP 6: Convert Discount from "5%" to Decimal

In [0]:
from pyspark.sql.functions import regexp_replace

silver_order_df = silver_order_df.withColumn(
    "discount",
    when(col("discount").like("5%"), 
           (regexp_replace("discount", "5%", "0.05"))
    ).when(col("discount").like("10%"), 
           (regexp_replace("discount", "10%", "0.1"))
    ).otherwise(col("discount"))
)

silver_order_df.display()

# 👨‍🏫 STEP 7: Remove Negative Quantity

In [0]:

silver_order_df = silver_order_df.filter(col("quantity")>=0)

In [0]:
silver_order_df.display()

# STEP 8: Cast unit_price to Decimal

In [0]:
from pyspark.sql.types import DecimalType

silver_order_df = silver_order_df.withColumn(
    "unit_price",
    col("unit_price").try_cast(DecimalType(10,2))
)
silver_order_df.display()

# 👨‍🏫 STEP 9: Recalculate Total Amount

In [0]:

silver_order_df = silver_order_df.withColumn(
    "total_amount",
    col("quantity").cast("int") * col("unit_price") * (1 - col("discount").cast("double"))
)

silver_order_df.display()

In [0]:
silver_order_df.write.format("delta").mode("overwrite").saveAsTable("retailproject.silver.silver_orders")

# Retail Customers – Silver Layer Cleaning | Databricks End-to-End Project

In [0]:

customers_bronze_df = spark.table("retailproject.bronze.bronze_customers")

display(customers_bronze_df)

# Step2 - Trim the columns

In [0]:

from pyspark.sql.functions import trim, col

customers_df = customers_bronze_df.select(
    trim(col("customer_id")).alias("customer_id"),
    trim(col("customer_name")).alias("customer_name"),
    trim(col("email")).alias("email"),
    trim(col("phone")).alias("phone"),
    trim(col("gender")).alias("gender"),
    trim(col("date_of_birth")).alias("date_of_birth"),
    trim(col("city")).alias("city"),
    trim(col("state")).alias("state"),
    trim(col("registration_date")).alias("registration_date"),
    trim(col("loyalty_points")).alias("loyalty_points"),
    trim(col("status")).alias("status"),
    col("ingest_time")
)

display(customers_df)

# 👨‍🏫 STEP 3: Remove Duplicate Customers

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_spec = Window.partitionBy("customer_id") \
                    .orderBy(desc("ingest_time"))

customers_df = customers_df.withColumn(
    "row_num",
    row_number().over(window_spec)
)

customers_df = customers_df.filter("row_num = 1") \
                           .drop("row_num")

customers_df.display()

#  STEP 4: Fix and Standardize Status

In [0]:
from pyspark.sql.functions import upper, when

customers_df = customers_df.withColumn(
    "status",
    upper(col("status"))
)

customers_df = customers_df.withColumn(
    "status",
    when(col("status").isin("ACTIVE", "INACTIVE"), col("status"))
    .otherwise("UNKNOWN")
)

customers_df.display()

# 👨‍🏫 STEP 5: Convert loyalty_points to Integer


In [0]:

from pyspark.sql.functions import regexp_replace

customers_df = customers_df.withColumn(
    "loyalty_points",
    regexp_replace(col("loyalty_points"), "[^0-9]", "")
)

customers_df = customers_df.withColumn(
    "loyalty_points",
    col("loyalty_points").try_cast("int")
)
     

In [0]:
customers_df.display()

# 👨‍🏫 STEP 6: Validate Email Format

In [0]:
customers_df = customers_df.withColumn(
    "email_valid",
    col("email").rlike("^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
)

# Example: meenadas@email.com - valid, karansingh@email - invalid

# customers_df.select("email", "email_valid").display()

customers_df = customers_df.filter(col("email_valid") == True)

In [0]:
from pyspark.sql.functions import lower, trim

customers_df = customers_df.withColumn(
    "email",
    lower(trim(col("email")))
)

customers_df.display()

# 👨‍🏫 STEP 7: Fix Date Formats

In [0]:
from pyspark.sql.functions import to_date, coalesce

customers_df = customers_df.withColumn(
    "dob_parsed",
    coalesce(
            try_to_date(col('date_of_birth'),'yyyy-MM-dd'),
            try_to_date(col('date_of_birth'),'dd-MM-yyyy'),
            try_to_date(col('date_of_birth'),'yyyy/MM/dd'),
            try_to_date(col('date_of_birth'),'dd/MM/yyyy')
        )
)

customers_df = customers_df.withColumn(
    "registration_parsed",
    coalesce(
            try_to_date(col('registration_date'),'yyyy-MM-dd'),
            try_to_date(col('registration_date'),'dd-MM-yyyy'),
            try_to_date(col('registration_date'),'yyyy/MM/dd'),
            try_to_date(col('registration_date'),'dd/MM/yyyy')
        )
)

customers_df.display()

# 👨‍🏫 STEP 8: Standardize Gender

In [0]:
customers_df = customers_df.withColumn(
    "gender",
    upper(col("gender"))
)

customers_df = customers_df.withColumn(
    "gender",
    when(col("gender") == "MALE", "M")
    .when(col("gender") == "FEMALE", "F")
    .otherwise("U")
)

In [0]:
customers_df.display()

In [0]:
customers_df.write.format("delta").mode("overwrite").saveAsTable("retailproject.silver.silver_customers")

In [0]:
customers_df = spark.read.table("retailproject.silver.silver_customers")
orders_df = spark.read.table("retailproject.silver.silver_orders")

customers_df.display()
orders_df.display()

In [0]:

dim_customer = customers_df.select("customer_id",
                    "customer_name",
                    "email",
                    "city",
                    "state",
                    "loyalty_points",
                    "status"                    
                    )
dim_customer.display()

In [0]:
dim_customer.write.format('delta').mode('overwrite').saveAsTable('retailproject.gold.gold_dim_customer')

In [0]:
fact_sales = orders_df.join(
    customers_df,
    orders_df.customer_id == customers_df.customer_id,
    "left"
).select(
    orders_df.order_id,
    orders_df.order_date,
    orders_df.customer_id,
    customers_df.customer_name,
    orders_df.product_id,
    orders_df.product_name,
    orders_df.category,
    orders_df.quantity,
    orders_df.unit_price,
    orders_df.discount,
    orders_df.total_amount,
    customers_df.city,
    customers_df.state,
    customers_df.gender,
    customers_df.loyalty_points
)

# fact_sales.display()


from pyspark.sql.functions import year , month , dayofmonth

order_enriched = fact_sales.\
    withColumn("Year",year("order_date")).\
    withColumn("Month",month("order_date")).\
    withColumn("Day",dayofmonth("order_date"))

order_enriched.display()



In [0]:
from pyspark.sql.functions import sum, countDistinct, avg, min,max

gold_df = order_enriched.groupBy(
    "product_id",
    "product_name",
    "customer_id",
    "customer_name",
    "category",
    "Year",
    "Month",
    "order_date",
    "gender",
    "city",
    "state",
    "loyalty_points"
).agg(
    sum("total_amount").alias("Total_revenue"),
    sum("quantity").alias("Total_quantity"),
    countDistinct("order_id").alias("Total_orders"),
    avg("total_amount").alias("Avg_revenue"),
    avg("quantity").alias("Avg_quantity"),
    avg("loyalty_points").alias("Avg_loyalty_points"),
    min("unit_price").alias("Min_price"),
    max("unit_price").alias("Max_price")
)
gold_df.display()

In [0]:
gold_df.write.format('delta').mode('overwrite').saveAsTable('retailproject.gold.gold_fact_Sales')